# **Comprehensive NLP Lab: From Preprocessing to Feature Extraction**

In this lab, you will explore a wide range of Natural Language Processing (NLP) techniques, from basic text preprocessing to advanced feature extraction and analysis. By the end of this lab, you will be able to:

1. **Tokenize** and preprocess text data.
2. Remove **stop words** and **punctuation**.
3. Apply **stemming** and **lemmatization**.
4. Extract features using **Bag of Words (BoW)** and **TF-IDF**.
5. Generate **n-grams** to capture contextual information.
6. Evaluate the impact of different preprocessing techniques on text data.

Let's dive in!

## **1. Setup the Environment**


Before we begin, ensure you have the necessary libraries installed. Run the following cell to install them:


In [1]:
# --- Install dependencies -------------------------------------------------
# nltk        -> tokenisation, stop words, stemmers, lemmatizer, corpora
# scikit-learn -> CountVectorizer / TfidfVectorizer for feature extraction
# pandas / matplotlib -> data handling and plotting
# The line is commented because the packages are normally installed already.
#!pip install nltk scikit-learn pandas matplotlib


Now, import the required libraries:

In [2]:
# --- Imports --------------------------------------------------------------
# nltk primitives: stopword list, word tokenizer, Porter stemmer,
# WordNet lemmatizer, and POS tagger. sklearn vectorizers turn text into
# numeric feature matrices (BoW and TF-IDF).

import nltk
import numpy as np
import re
import string
import pandas as pd
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [3]:
# --- Download NLTK data packages -----------------------------------------
# punkt / punkt_tab               -> sentence & word tokenizer models
# stopwords                        -> language-specific stop-word lists
# averaged_perceptron_tagger       -> part-of-speech tagger model
# wordnet                          -> lexical database used by the lemmatizer
# Download NLTK datasets
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sisaz\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sisaz\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\sisaz\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sisaz\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sisaz\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\sisaz\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_percep

True

## **2. Text Preprocessing**

### **Exercise 1: Tokenization and Stop Word Removal**

Tokenize the following text

In [4]:
# --- Exercise 1: Tokenization --------------------------------------------
# word_tokenize splits a raw string into linguistically meaningful tokens
# (words + punctuation). Notice how '(', ')', '!' and '.' become their own
# tokens -- this matters for the next cleaning step.
text = "Natural Language Processing (NLP) is a fascinating field of study! It involves analyzing and understanding human language."
tokenized_docs = word_tokenize(text)
for doc in tokenized_docs:
    print(doc)


Natural
Language
Processing
(
NLP
)
is
a
fascinating
field
of
study
!
It
involves
analyzing
and
understanding
human
language
.


Remove stop words and store the result in a variable called `filtered_tokens`

In [5]:
# --- Stop-word removal ----------------------------------------------------
# Stop words are very frequent grammatical words ('the', 'is', 'a', ...)
# that usually carry little topical information. We load the English list
# as a set (O(1) lookup) and keep only tokens that are NOT in the set.
stop_words = set(stopwords.words('english'))
print(f"The stop words are {stop_words}")
filtered_tokens = []
for word in tokenized_docs:
    if word not in stop_words:
        filtered_tokens.append(word)


The stop words are {'yourselves', 'been', "it'll", 'only', 'does', "don't", 'or', 'each', 'themselves', "i'd", "couldn't", 'is', "wasn't", 'theirs', 'because', "he's", 'same', 's', 'she', 'up', 'both', 'doing', 'them', 'now', 'then', 'those', 'y', 'own', 'off', 'when', "they're", "you're", "she's", 'we', "they'd", 'are', 'of', 'has', "should've", "we're", 'that', 'wouldn', 'mustn', "shouldn't", 'having', 'from', "it'd", "you'd", 'had', 'his', 'your', "we'll", 'couldn', 'a', 'm', 'it', 'few', 'at', 'hers', 'with', "shan't", 'me', 'was', 'their', 'being', 'himself', 'any', 'can', 'into', 'wasn', 'will', 'as', "we'd", "they'll", 'after', 'do', 'did', "i'll", 'during', 'her', 'an', 'haven', 'whom', 'too', 'this', 't', "she'll", 'but', "they've", 'who', 'out', 'down', "she'd", 'the', 'here', "you've", 'be', 'don', "hasn't", "aren't", 'ours', "he'll", 'll', 'over', 'shan', 'on', 'ma', 'some', 'where', 'above', 'again', 'before', 'doesn', 're', 'mightn', 'so', "mustn't", 'ain', 'other', 'myse

In [6]:
# Print the cleaned tokens. Punctuation still remains; we will remove it
# in the custom pipeline (Exercise 6).
print("Filtered Tokens:", filtered_tokens)

Filtered Tokens: ['Natural', 'Language', 'Processing', '(', 'NLP', ')', 'fascinating', 'field', 'study', '!', 'It', 'involves', 'analyzing', 'understanding', 'human', 'language', '.']


### **Exercise 2: Stemming and Lemmatization**

Apply stemming and lemmatization to the `filtered_tokens`. Compare the results.

In [7]:
# --- Exercise 2: Prepare stemmer & lemmatizer ----------------------------
# PorterStemmer:     rule-based chopping of suffixes ('studies' -> 'studi').
# WordNetLemmatizer: dictionary-based reduction to the canonical lemma
#                    ('studies' -> 'study'). Lemmatization is slower but
#                    produces real words.
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

Apply stemming and store the result in `stemmed_tokens`

In [8]:
# Apply the Porter stemmer to every filtered token. Stemming is aggressive:
# 'fascinating' -> 'fascin', 'language' -> 'languag' (not real words).
stemmed_tokens = []
for word in filtered_tokens:
    stemmed_tokens.append(stemmer.stem(word))
    #stemmed_tokens.append(snowball.stem(word)) # requires 'corpora/wordnet' -> nltk.download()


In [9]:
# Inspect the stemmed output.
print("Stemmed Tokens:", stemmed_tokens)

Stemmed Tokens: ['natur', 'languag', 'process', '(', 'nlp', ')', 'fascin', 'field', 'studi', '!', 'it', 'involv', 'analyz', 'understand', 'human', 'languag', '.']


Apply lemmatization and store the result in `lemmatized_tokens`

In [10]:
# Apply the lemmatizer on top of the already-stemmed tokens. Because the
# stems are no longer valid dictionary words, the lemmatizer cannot improve
# them -- this demonstrates why lemmatization should be applied to RAW
# tokens, not to stems (see Exercise 6 for the correct order).
lemmatized_tokens = []
for word in stemmed_tokens:
    lemmatized_tokens.append(lemmatizer.lemmatize(word))


In [11]:
# Inspect the lemmatized output.
print("Lemmatized Tokens:", lemmatized_tokens)

Lemmatized Tokens: ['natur', 'languag', 'process', '(', 'nlp', ')', 'fascin', 'field', 'studi', '!', 'it', 'involv', 'analyz', 'understand', 'human', 'languag', '.']


## **3. Feature Extraction**

### **Exercise 3: Bag of Words (BoW)**

Use the `CountVectorizer` from `scikit-learn` to create a Bag of Words representation of the following corpus

In [12]:
# --- Exercise 3: Toy corpus for feature extraction -----------------------
# Three short documents. We will represent each of them numerically using
# Bag of Words, TF-IDF and bigrams.
corpus = [
    "I love NLP.",
    "NLP is amazing.",
    "I enjoy learning new things in NLP."
]

In [13]:
# --- Bag of Words ---------------------------------------------------------
# CountVectorizer builds a vocabulary from the corpus and returns a
# document-term matrix where cell (i, j) = count of word j in document i.
# max_features=1000 keeps only the 1000 most frequent words (not a
# constraint here since the vocabulary is tiny).
# Step 1: Initialize the CountVectorizer
#let's take only the most common 1000 words
from sklearn.feature_extraction.text import CountVectorizer
bow_vect = CountVectorizer(max_features=1000)

# Step 2: Fit and transform the corpus into a BoW representation
# fit creates one entry for each different word seen
X = bow_vect.fit_transform(corpus).toarray()

In [14]:
# The rows are documents, the columns are words from the learned vocabulary.
# 'i', 'a' and single-character tokens are dropped by the default token
# pattern (it requires >=2 word characters).
print("Bag of Words:\n", X)
print("Vocabulary:", bow_vect.get_feature_names_out())

Bag of Words:
 [[0 0 0 0 0 1 0 1 0]
 [1 0 0 1 0 0 0 1 0]
 [0 1 1 0 1 0 1 1 1]]
Vocabulary: ['amazing' 'enjoy' 'in' 'is' 'learning' 'love' 'new' 'nlp' 'things']


### **Exercise 4: TF-IDF**

Use the `TfidfVectorizer` from `scikit-learn` to create a TF-IDF representation of the same corpus. Store the result in `X_tfidf`

In [15]:
# --- Exercise 4: TF-IDF ---------------------------------------------------
# TF-IDF weights each word by:
#   TF  = how often the word appears in THIS document
#   IDF = log(N / df) -> penalises words appearing in many documents
# Common words ('nlp' here) therefore get a lower weight than rare ones.
# min_df=1 keeps every word that appears in at least one document.
# your code here
# Step 1: Initialize the TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(min_df = 1)

# Step 2: Fit and transform the corpus into a TF-IDF representation
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)


for doc,tf_idf_doc in zip(corpus,tfidf_matrix.todense()):
    print("DOC:", doc)
    print(np.around(tf_idf_doc,3)) # El tres define los decimales
    print()

DOC: I love NLP.
[[0.    0.    0.    0.    0.    0.861 0.    0.509 0.   ]]

DOC: NLP is amazing.
[[0.652 0.    0.    0.652 0.    0.    0.    0.385 0.   ]]

DOC: I enjoy learning new things in NLP.
[[0.    0.432 0.432 0.    0.432 0.    0.432 0.255 0.432]]



In [16]:
# Full dense TF-IDF matrix and its vocabulary. Each row is L2-normalised.
print("TF-IDF:\n", tfidf_matrix.toarray())
print("Vocabulary:", tfidf_vectorizer.get_feature_names_out())

TF-IDF:
 [[0.         0.         0.         0.         0.         0.861037
  0.         0.50854232 0.        ]
 [0.65249088 0.         0.         0.65249088 0.         0.
  0.         0.38537163 0.        ]
 [0.         0.43238509 0.43238509 0.         0.43238509 0.
  0.43238509 0.2553736  0.43238509]]
Vocabulary: ['amazing' 'enjoy' 'in' 'is' 'learning' 'love' 'new' 'nlp' 'things']


### **Exercise 5: N-grams**

Generate `bigrams (2-grams)` from the corpus using `CountVectorizer`. Store the result in `X_bigram`

In [17]:
# --- Exercise 5: Bigrams --------------------------------------------------
# ngram_range=(2, 2) tells CountVectorizer to use only pairs of consecutive
# words as features. Bigrams capture short-range context that unigrams miss
# (e.g. 'new york' or 'not good').
import nltk
from nltk.util import bigrams
# Step 1: Initialize the CountVectorizer with ngram_range=(2, 2)
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))

# Step 2: Fit and transform the corpus into a bigram representation
X_bigram = bigram_vectorizer.fit_transform(corpus)


In [18]:
# Show the bigram document-term matrix and the bigram vocabulary.
print("Bigrams:\n", X_bigram.toarray())
print("Bigram Vocabulary:", bigram_vectorizer.get_feature_names_out())

Bigrams:
 [[0 0 0 0 1 0 0 0]
 [0 0 1 0 0 0 1 0]
 [1 1 0 1 0 1 0 1]]
Bigram Vocabulary: ['enjoy learning' 'in nlp' 'is amazing' 'learning new' 'love nlp'
 'new things' 'nlp is' 'things in']


## **4. Advanced Exercise: Custom Preprocessing Pipeline**

### **Exercise 6: Build a Custom Preprocessing Pipeline**

Combine all the preprocessing steps (tokenization, stop word removal, punctuation removal, stemming/lemmatization) into a single function. 

In [19]:
# --- Exercise 6: End-to-end preprocessing pipeline -----------------------
# Combines the four canonical cleaning steps in the correct order:
#   1. Lower-case + tokenise
#   2. Drop stop words
#   3. Drop punctuation characters
#   4. Lemmatise (on raw tokens, NOT on stems)
# The function returns a clean list of tokens ready for vectorisation.
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# nltk.download('punkt_tab')
# nltk.download('stopwords')
# nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def text_preprocessing_pipeline(text):
    # Step 1: Tokenize the text
    tokens = word_tokenize(text.lower())

    # Step 2: Remove stop words
    tokens = [t for t in tokens if t not in stop_words]

    # Step 3: Remove punctuation
    tokens = [t for t in tokens if t not in string.punctuation]

    # Step 4: Apply lemmatization
    lemmatized_tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return lemmatized_tokens

Apply this function to the following text

In [20]:
# Run the pipeline on a sample sentence. Compare with Exercise 1 to see the
# effect of lower-casing, punctuation stripping and lemmatisation.
text = "Natural Language Processing (NLP) is a fascinating field of study! It involves analyzing and understanding human language."

processed_text = text_preprocessing_pipeline(text)


In [21]:
# Display the final cleaned token list.
print("Processed Text:", processed_text)

Processed Text: ['natural', 'language', 'processing', 'nlp', 'fascinating', 'field', 'study', 'involves', 'analyzing', 'understanding', 'human', 'language']


## **5. Evaluation of Preprocessing Techniques**

### **Exercise 7: Compare Preprocessing Techniques**

Compare the results of stemming and lemmatization on the following sentence. Store the results in `stemmed_tokens` and `lemmatized_tokens`

In [22]:
# --- Exercise 7: Stemming vs Lemmatization comparison --------------------
# Same input, two different normalisation strategies. Key observation:
#   'cats'   -> stem 'cat'   / lemma 'cat'   (both agree)
#   'mice'   -> stem 'mice'  / lemma 'mouse' (lemmatizer wins here)
# Lemmatization understands irregular plurals; stemming does not.
sentence = "The cats are playing with the mice in the garden."

lemmatizer = WordNetLemmatizer()

# your code here
# Step 1: Tokenize and preprocess the sentence and store the result in filtered_tokens
filtered_tokens = word_tokenize(sentence.lower())

# Step 2: Apply stemming
stemmer = PorterStemmer()

stemmed_tokens = []
for word in filtered_tokens:
    stemmed_tokens.append(stemmer.stem(word))

# Step 3: Apply lemmatization
lemmatizer = WordNetLemmatizer()
lemmatized_tokens = []
for word in stemmed_tokens:
    lemmatized_tokens.append(lemmatizer.lemmatize(word))



In [23]:
# Print the three versions side by side to see the differences clearly.
print("Original Tokens:", filtered_tokens)
print("Stemmed Tokens:", stemmed_tokens)
print("Lemmatized Tokens:", lemmatized_tokens)

Original Tokens: ['the', 'cats', 'are', 'playing', 'with', 'the', 'mice', 'in', 'the', 'garden', '.']
Stemmed Tokens: ['the', 'cat', 'are', 'play', 'with', 'the', 'mice', 'in', 'the', 'garden', '.']
Lemmatized Tokens: ['the', 'cat', 'are', 'play', 'with', 'the', 'mouse', 'in', 'the', 'garden', '.']


## **6. Real-World Dataset: Sentiment Analysis**

### **Exercise 8: Preprocess and Analyze Tweets**

In this exercise, you will work with a real-world dataset of tweets. The dataset contains 5000 positive and 5000 negative tweets. Your task is to preprocess the tweets and extract features for sentiment analysis.


In [24]:
# Download the NLTK twitter_samples corpus (5k positive + 5k negative).
nltk.download('twitter_samples')

[nltk_data] Downloading package twitter_samples to
[nltk_data]     C:\Users\sisaz\AppData\Roaming\nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!


True

In [25]:
# Import the corpus reader.
# Load the dataset
from nltk.corpus import twitter_samples

Load the dataset of positive and negative tweets. 

In [26]:
# Load the raw tweet strings from the two JSON files shipped with NLTK.
positive_tweets = twitter_samples.strings('positive_tweets.json')
negative_tweets = twitter_samples.strings('negative_tweets.json')

Combine them into a single list called ``all_tweets`` and create a corresponding list of labels called `labels`.

In [27]:
# Build the full dataset: concatenate the two tweet lists and create a
# parallel label vector (1 = positive, 0 = negative). This is the setup
# used for supervised sentiment classification.
# your code here

# Combine the datasets
all_tweets = positive_tweets + negative_tweets
labels = [1] * len(positive_tweets) + [0] * len(negative_tweets)



In [28]:
# Sanity-check: print the first tweet and its label.
# Print a sample tweet
print("Sample Tweet:", all_tweets[0])
print("Label:", labels[0])

Sample Tweet: #FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)
Label: 1


### **Exercise 9: Preprocess Tweets**

Apply the custom preprocessing pipeline to the entire dataset of tweets. Store the result in ``preprocessed_tweets``.

In [29]:
# Run the custom pipeline on every tweet. The result is a list of
# token lists -- one cleaned tweet per row.
# Step 1: Apply the preprocessing pipeline to all tweets
# your code here
preprocessed_tweets = []
for tweet in all_tweets:
    tweet = text_preprocessing_pipeline(tweet)
    preprocessed_tweets.append(tweet)


In [30]:
# Inspect a cleaned tweet to confirm the pipeline worked.
# Print a sample preprocessed tweet
print("Preprocessed Tweets Sample:", preprocessed_tweets[0])

Preprocessed Tweets Sample: ['followfriday', 'france_inte', 'pkuchly57', 'milipol_paris', 'top', 'engaged', 'member', 'community', 'week']


### **Exercise 10: Feature Extraction on Tweets**

Extract features from the preprocessed tweets using **Bag of Words** and **TF-IDF**. Store the results in ``X_bow`` and ``X_tfidf``, respectively.

In [31]:
# --- Exercise 10: Feature extraction on the tweet corpus -----------------
# Vectorizers expect strings, not lists of tokens, so we join each cleaned
# tweet back into a space-separated string. Then we build a Bag-of-Words
# matrix limited to the 1000 most frequent tokens -- a typical first
# feature set for a baseline sentiment classifier.
preprocessed_texts = [" ".join(tokens) for tokens in preprocessed_tweets]

# Step 1: Create a Bag of Words representation
bow_vect = CountVectorizer(max_features=1000)


# Step 2: Create a TF-IDF representation
X = bow_vect.fit_transform(preprocessed_texts).toarray()
print("Bag of Words:\n", X)
print("Vocabulary:", bow_vect.get_feature_names_out())


Bag of Words:
 [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
Vocabulary: ['000' '10' '100' '11' '12' '13' '15' '20' '2015' '24' '2nd' '30' '40'
 '50' '5sos' '969horan696' 'able' 'absolutely' 'acc' 'account' 'act'
 'active' 'actually' 'add' 'added' 'address' 'adeccowaytowork' 'af'
 'afternoon' 'ago' 'agree' 'ah' 'ahh' 'ai' 'aint' 'air' 'airport' 'al'
 'album' 'all' 'allah' 'allowed' 'almost' 'alone' 'along' 'already'
 'alright' 'also' 'always' 'amazing' 'amber' 'amp' 'ang' 'annoying'
 'another' 'answer' 'anymore' 'anyone' 'anything' 'anyway' 'app'
 'apparently' 'appreciate' 'appreciated' 'aqui' 'around' 'arrived' 'art'
 'article' 'as' 'ask' 'asked' 'asking' 'asleep' 'ate' 'august' 'australia'
 'available' 'aw' 'awake' 'away' 'awesome' 'awful' 'aww' 'awww' 'awwww'
 'babe' 'baby' 'back' 'bad' 'badly' 'bae' 'bag' 'ball' 'bam' 'bank'
 'barsandmelody' 'bb' 'bby' 'bc' 'bday' 'beach' 'beat' 'beautiful'
 'beauty' 'become'

## **7. Conclusion**

In this lab, you explored a wide range of NLP techniques, from basic text preprocessing to advanced feature extraction and analysis. You also worked with a real-world dataset of tweets and applied your knowledge to preprocess and extract features for sentiment analysis.